In [0]:
%pip install databricks-labs-dqx

In [0]:
from databricks.labs.dqx.profiler.profiler import DQProfile
from databricks.labs.dqx.profiler.dlt_generator import DQDltGenerator

def reverse_generator(col_name, arguments, func, name=""):
    if not name:
        name = "_".join(("col", col_name, func))

    if func == "is_not_null":
        return DQProfile(func, col_name, name)

    elif func == "is_in":
        return DQProfile(func, col_name, name, parameters={"in": arguments["allowed"]})

    elif func == "is_in_range":
        return DQProfile(
            "min_max",
            col_name,
            name,
            parameters={"min": arguments["min_limit"], "max": arguments["max_limit"]},
        )

    elif func == "is_not_null_and_not_empty":
        return DQProfile(
            "is_not_null_or_empty",
            col_name,
            name,
            parameters={"trim_strings": arguments.get("trim_strings", True)},
        )

def transfrom_checks_dlt(checks):

    allowed_list = [
    "is_not_null",
    "is_in",
    "is_in_range",
    "is_not_null_and_not_empty",
    "sql_expression",
    ]
    profile_list = []
    sql_expressions = {}

    for check in checks:
        func = check["check"]["function"]
        if func not in allowed_list:
            print(
                f"Warning: Function '{func}' is not a recognized DQX DLTGenerator mapping and is not 'sql_expression'. Skipping this check."
            )
            continue

        col_names_set = set()
        arguments = check["check"]["arguments"]
        check_name = check.get("name", "")

        if func == "sql_expression":
            if not check_name:
                raise AttributeError(
                    "sql_expression check requires a 'name' for usage in dlt"
                )
            sql_expressions.update(
                {check["name"]: check["check"]["arguments"]["expression"]}
            )

        if "col_names" in arguments and isinstance(arguments["col_names"], list):
            col_names_set.update(arguments["col_names"])

        if "col_name" in arguments:
            col_names_set.add(arguments["col_name"])

        for col_name in col_names_set:
            profile_list.append(reverse_generator(col_name, arguments, func, check_name))


    sql_expressions.update(
        DQDltGenerator(None).generate_dlt_rules(profile_list, language="python_dict")
    )
    return sql_expressions




In [0]:
checks_yaml="""
- check:
    arguments:
      col_names: [id,revenue,is_active]
    function: is_not_null
  criticality: error
- check:
    arguments:
      col_name: city
    function: is_not_null_and_not_empty
  criticality: error
- check:
    arguments:
      col_name: revenue
      limit: 0
    function: is_not_less_than
  criticality: error
  name: revenue_is_positive
- check:
    arguments:
      col_name: revenue
      min_limit: 0
      max_limit: 999
    function: is_in_range
- check:
    function: sql_expression
    arguments:
        expression: is_active is True
  criticality: error
  name: is_active_is_true
"""

In [0]:
import dlt

from pyspark.sql.functions import col
from pyspark.sql.functions import expr
import yaml

rules = transfrom_checks_dlt(yaml.safe_load(checks_yaml))
quarantine_rules = "NOT({0})".format(" AND ".join(rules.values()))


@dlt.view
def test_data_view():
    data = [
        (1, 30, True, "Cologne"),
        (2, 100, True, "Berlin"),
        (3, 50, False, "Munich"),
        (4, 75, True, "Hamburg"),
        (5, 120, True, "Frankfurt"),
        (6, -90, False, "Stuttgart"),
        (7, -60, True, "Dusseldorf"),
        (8, 80, False, "Dortmund"),
        (9, 110, True, "Essen"),
        (10, 150, True, "Leipzig"),
        (11, 40, False, "Bremen"),
        (12, 70, True, "Dresden"),
        (13, 95, True, "Hanover"),
        (14, 130, False, "Nuremberg"),
        (15, 85, True, "Duisburg"),
        (16, 55, False, "Bochum"),
        (17, 65, True, "Wuppertal"),
        (18, 105, True, "Bielefeld"),
        (19, 140, False, "Bonn"),
        (20, 1250, True, ""),
    ]
    columns = ["id", "revenue", "is_active", "city"]

    return spark.createDataFrame(data, columns)


@dlt.table
@dlt.expect_all_or_drop(rules)
def test_dqx_table():
    return spark.read.table("test_data_view")


@dlt.table
def quarantine_dqx_table():
    return spark.read.table("test_data_view").filter(expr(quarantine_rules))